In [5]:
import pandas as pd
import numpy as np
import seaborn as sns
import csv
import matplotlib.pyplot as plt
import optuna

from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
from sklearn.model_selection import GridSearchCV
from sklearn.model_selection import cross_val_score
from sklearn.metrics import roc_auc_score
from sklearn.preprocessing import StandardScaler
from sklearn.preprocessing import OneHotEncoder
from sklearn.preprocessing import OrdinalEncoder
from sklearn.utils import resample
from sklearn.feature_selection import SelectKBest, f_classif
from sklearn.neighbors import KNeighborsClassifier
from sklearn.ensemble import RandomForestClassifier
from optbinning import BinningProcess
from tqdm import tqdm
from pandas.api.types import is_numeric_dtype

from catboost import CatBoostClassifier

TARGET = 'target'
ID_COL = 'reco_id_curr'
RANDOM_STATE = 42

def split_xy(frame):
    y = frame[TARGET].astype(int)
    X = frame.drop(columns=[TARGET, ID_COL], errors='ignore')
    return X, y

def gini(y_true, y_pred):
    auc = roc_auc_score(y_true, y_pred)
    return 2 * auc  - 1

# Добавление колонок, разбиение, encoding, scaling, splitting

In [6]:
data = pd.read_csv('data_clear.csv')

f = 1

In [7]:
# Создание новых колонок
if (f):
    data['ability_to_pay'] = data['annuity_payment'] / data['income']
    data['blow_to_the_budget'] = abs(data['goods_price'] - data['loan_body'])
    data.drop(columns=['goods_price'])
    data['days_birth'] = -data['days_birth'] / 365
    data['working_person'] = (-data['days_employed'] / 365) / data['days_birth']
    f = 0

In [8]:
# RANDOM_STATE = 42
# Сначала отделяем 10% для late_test
train_val_test, data_late = train_test_split(
    data,
    test_size=0.1,  # 10% на late_test
    stratify=data['target'],
    random_state=RANDOM_STATE
)

# Из оставшихся 90% берем 70% для train (это ~77.8% от 90%)
# train_size = 0.7 / 0.9 ≈ 0.7778
data_train, val_test = train_test_split(
    train_val_test,
    train_size=0.7778,  # 70% от ВСЕХ данных
    stratify=train_val_test['target'],
    random_state=RANDOM_STATE
)

# Из оставшихся 20% делим поровну на val и test
data_val, data_test = train_test_split(
    val_test,
    test_size=0.5,  # 10% от ВСЕХ данных
    stratify=val_test['target'],
    random_state=RANDOM_STATE
)

In [9]:
# Бьём данные на X и y
data_train_sampled = resample(data_train, n_samples=50_000, replace=False, random_state=RANDOM_STATE)

X_train, y_train = split_xy(data_train_sampled)
X_val, y_val = split_xy(data_val)
X_test, y_test = split_xy(data_test)
X_late, y_late = split_xy(data_late)

In [10]:
categorical_columns = X_train.select_dtypes(include=['str']).columns

columns_for_ord_enc = ['type_of_occupation', 'type_of_organization']
columns_for_ohe =  [col for col in categorical_columns if (col not in columns_for_ord_enc)]

In [11]:
# Encoding
ohe = OneHotEncoder(sparse_output=False, handle_unknown='ignore')
ord_enc = OrdinalEncoder(handle_unknown='use_encoded_value', unknown_value=-1)

num_cols = X_train.select_dtypes(include=[np.number]).columns.tolist()
cat_cols = [c for c in X_train.columns if c not in num_cols]

def process_split(X, fit=False):
    num_part = X[num_cols]
    cat_parts = X[cat_cols]
    
    ohe_array = ohe.fit_transform(X[columns_for_ohe]) if fit else ohe.transform(X[columns_for_ohe])
    ohe_df = pd.DataFrame(ohe_array, columns=ohe.get_feature_names_out(columns_for_ohe), index=X.index)
    
    ord_array = ord_enc.fit_transform(X[columns_for_ord_enc]) if fit else ord_enc.transform(X[columns_for_ord_enc])
    ord_df = pd.DataFrame(ord_array, columns=ord_enc.get_feature_names_out(columns_for_ord_enc), index=X.index)
    return pd.concat([ohe_df, ord_df, num_part], axis=1)

X_train_enc = process_split(X_train, fit=True)
X_val_enc = process_split(X_val, fit=False)
X_test_enc = process_split(X_test, fit=False)
X_late_enc = process_split(X_late, fit=False)

In [12]:
# Scaling
scaler = StandardScaler()
X_train_enc_trans = scaler.fit_transform(X_train_enc)
X_val_enc_trans = scaler.transform(X_val_enc)
X_test_enc_trans = scaler.transform(X_test_enc)
X_late_enc_trans = scaler.transform(X_late_enc)


X_train_enc_trans = pd.DataFrame(X_train_enc_trans, columns=X_train_enc.columns)
X_val_enc_trans = pd.DataFrame(X_val_enc_trans, columns=X_val_enc.columns)
X_test_enc_trans = pd.DataFrame(X_test_enc_trans, columns=X_test_enc.columns)
X_late_enc_trans = pd.DataFrame(X_late_enc_trans, columns=X_late_enc.columns)

In [13]:
model = LogisticRegression(random_state=RANDOM_STATE)

model.fit(X_train_enc_trans, y_train)

probabilities = model.predict_proba(X_val_enc_trans)[:,1]
gini(y_val, probabilities)

0.45079550459561113

In [25]:
# Создание таблиц по IV и F Score

def feature_scoring(col: str) -> dict:
    result = {'feature': col, 'IV': 0.0, 'F_score': 0.0, 'p_value': np.nan}
    X_one = X_train_enc_trans[[col]]
    y_one = y_train
    if X_one[col].nunique() < 2:
        return result

    f_sel = SelectKBest(score_func=f_classif, k='all')
    f_sel.fit(X_one, y_one)
    result['F_score'] = float(f_sel.scores_[0])
    result['p_value'] = float(f_sel.pvalues_[0])

    bp = BinningProcess(variable_names=[col])
    bp.fit(X_one, y_one)
    result['IV'] = float(bp.get_binned_variable(col).binning_table.iv)
    return result

rows = [feature_scoring(col) for col in tqdm(X_train_enc_trans.columns, desc='IV и F-score')]
scores = pd.DataFrame(rows)
iv_table = scores[['feature', 'IV']].sort_values('IV', ascending=False).reset_index(drop=True)
f_table = scores[['feature', 'F_score', 'p_value']].sort_values('F_score', ascending=False).reset_index(drop=True)


iv_table.to_csv('iv_table.csv', index=False)
f_table.to_csv('f_table.csv', index=False)

IV и F-score: 100%|██████████| 126/126 [00:04<00:00, 25.93it/s]


In [26]:
# Перебор топ сколько брать из каждой колнки (берём топ n из iv и f и объединяем)

def top_n_f(n):
    combined_set = set(f_table['feature'].tolist()[:n]) | set(iv_table['feature'].tolist()[:n])
    list(combined_set)
    return combined_set

# Подбор нужного n
    
top_n = [20, 35, 40, 45, 50]
top = []
for n in tqdm(range(25, 26)):
    cols = top_n_f(n)
    cols = pd.Index(cols)
    model.fit(X_train_enc_trans[cols], y_train)
    predictions = model.predict(X_val_enc_trans[cols])
    probabilities = model.predict_proba(X_val_enc_trans[cols])[:,1]
    top.append([n, len(cols), gini(y_val, probabilities)])

100%|██████████| 1/1 [00:00<00:00,  3.98it/s]


In [28]:
top.sort(key=lambda x: x[2], reverse=True)

cols = pd.Index(top_n_f(25))

X_train_enc_trans_cp = X_train_enc_trans
X_val_enc_trans_cp = X_val_enc_trans
X_test_enc_trans_cp = X_test_enc_trans
X_late_enc_trans_cp = X_late_enc_trans

X_train_enc_trans_cp = X_train_enc_trans_cp[cols]
X_val_enc_trans_cp = X_val_enc_trans_cp[cols]
X_test_enc_trans_cp = X_test_enc_trans_cp[cols]
X_late_enc_trans_cp = X_late_enc_trans_cp[cols]

In [ ]:
X_train_enc_trans_cp.to_csv('X_train.csv', index=False)
X_val_enc_trans_cp.to_csv('X_val.csv', index=False)
X_test_enc_trans_cp.to_csv('X_test.csv', index=False)
X_late_enc_trans_cp.to_csv('X_late.csv', index=False)

y_train.to_csv('y_train', index=False)
y_val.to_csv('y_val', index=False)
y_test.to_csv('y_test', index=False)
y_late.to_csv('y_late', index=False)

# Перебор гиперпараметров (закомментирован, ниже написаны лучшие (на самом деле не перебирали полностью))

In [33]:
# LOGREG_GRID = {
#     'C': [0.1, 1, 10, 100],
#     'class_weight': [None, 'balanced'],
#     'max_iter' : [1000],
#     'random_state' : [RANDOM_STATE]
# }

# KNN_GRID = {
#     'n_neighbors': [1500],
#     'weights': ['uniform', 'distance'],
#     'metric': ['euclidean', 'manhattan', 'minkowski'],
#     'algorithm': ['auto', 'ball_tree', 'kd_tree', 'brute']
# }

# RF_GRID = {
#     'n_estimators': [100, 200, 300],
#     'max_depth': [10, 15, 20],
#     'min_samples_leaf': [2, 5, 10, 20, 30],
#     'class_weight': [None, 'balanced'],
#     'random_state' : [RANDOM_STATE]
# }
# def optimize_hyperparameters(model_class, grid, X_train, y_train, X_val, y_val, n_trials=1, direction="maximize"):
#     for i in grid.values():
#         n_trials *= len(i)
#     def objective(trial):
#         params = {}
#         for key, values in grid.items():
#             if isinstance(values, list):
#                 params[key] = trial.suggest_categorical(key, values)
#             elif isinstance(values, tuple):
#                 if isinstance(values[0], int) and isinstance(values[1], int):
#                     params[key] = trial.suggest_int(key, values[0], values[1])
#                 else:
#                     params[key] = trial.suggest_float(key, values[0], values[1])
#         model = model_class(**params)
#         model.fit(X_train, y_train)
#         probabilities = model.predict_proba(X_val)[:,1]
#         return gini(y_val, probabilities)
    
#     study = optuna.create_study(direction=direction)
#     study.optimize(objective, n_trials=n_trials)
    
#     return study.best_params

# best_parameters_for_log_reg = optimize_hyperparameters(
#     model_class=LogisticRegression,
#     grid=LOGREG_GRID,
#     X_train=X_train_enc_trans_cp,
#     y_train=y_train,
#     X_val=X_val_enc_trans_cp,
#     y_val=y_val,
# )

# best_parameters_for_knn = optimize_hyperparameters(
#     model_class=KNeighborsClassifier,
#     grid=KNN_GRID,
#     X_train=X_train_enc_trans_cp,
#     y_train=y_train,
#     X_val=X_val_enc_trans_cp,
#     y_val=y_val,
# )

# best_parameters_for_rf = optimize_hyperparameters(
#     model_class=RandomForestClassifier,
#     grid=RF_GRID,
#     X_train=X_train_enc_trans_cp,
#     y_train=y_train,
#     X_val=X_val_enc_trans_cp,
#     y_val=y_val,
# )

In [34]:
# После перебора гиперпараметров выбраны лучшие
best_parameters_for_log_reg = {'C': 10.0, 'class_weight': 'balanced', 'max_iter' : 1000}
best_parameters_for_knn = { 'n_neighbors' : 1500 }
best_parameters_for_rf = {'n_estimators': 300, 'max_depth': 15, 'min_samples_leaf': 5, 'class_weight': None, 'random_state': 42}

In [35]:
model_LogReg = LogisticRegression(**best_parameters_for_log_reg)
model_KNN = KNeighborsClassifier(**best_parameters_for_knn)
model_RF = RandomForestClassifier(**best_parameters_for_rf)

model_LogReg.fit(X_train_enc_trans_cp, y_train)
probabilities = model_LogReg.predict_proba(X_late_enc_trans_cp)[:,1]
print("Logistic Regression.") 
print("Лучшие праметры: С = 10, class_weight = balanced, max_iter = 1000")
print("Gini:", gini(y_late, probabilities),"\n")


model_KNN.fit(X_train_enc_trans_cp, y_train)
probabilities = model_KNN.predict_proba(X_late_enc_trans_cp)[:,1]
print("K-Nearest Neighbors.") 
print("Лучшие праметры: n_neighbors = 1500")
print("Gini:", gini(y_late, probabilities),"\n")


model_RF.fit(X_train_enc_trans_cp, y_train)
probabilities = model_RF.predict_proba(X_late_enc_trans_cp)[:,1]
print("Random Forest.")
print("Лучшие праметры: n_estimators = 300, max_depth = 15, min_samples_leaf = 5, class_weight = None, random_state = 42")
print("Gini:", gini(y_late, probabilities))



Logistic Regression.
Лучшие праметры: С = 10, class_weight = balanced, max_iter = 1000
Gini: 0.47891332359465566 

K-Nearest Neighbors.
Лучшие праметры: n_neighbors = 1500
Gini: 0.4386667365391437 

Random Forest.
Лучшие праметры: n_estimators = 300, max_depth = 15, min_samples_leaf = 5, class_weight = None, random_state = 42
Gini: 0.4791958321872398
